# LlamaIndex 实现 RAG（最新版 · 2026-08）

> 基于 LlamaIndex **0.12–0.14.x** 重构，同步 2026 年最新推荐 API 与工程实践。
> 模型使用**阿里云百炼 DashScope 千问**（在线 API），第 1 节先填 Key 再跑后续单元格。

相比旧版的主要改动：
- 安装模块化，去掉 `fitz` / `frontend` 等无效包，`paddleocr` 降级为可选
- 模型换成**千问（DashScope）**，用 `getpass` 交互输入 API Key，不再硬编码空字符串
- 用 `Settings` 统一配置 LLM 与 Embedding 模型
- 检索升级为 **向量 + BM25 混合检索 + Rerank 重排**
- Agent 用新版 **`AgentWorkflow`** 替代已弃用的 `llm.predict_and_call`
- 修复旧示例里 chat_engine 引用未定义变量 `index` 的 bug
- 补充 `IngestionPipeline`、索引持久化等工程实践

In [11]:
# ══════════════════════════════════════════════════════════
# ══ 【cell-01】环境安装 (pip install) ══
# ══════════════════════════════════════════════════════════
# ============ 环境安装（建议 Python 3.11+，按需执行）============
# 说明：这些是 RAG 全流程需要的依赖，已按用途分组。
#       每一行都能单独跑，装过会自动跳过。

# —— 核心框架：LlamaIndex 主库（0.10+ 模块化拆包，只装需要的）——
!pip install -U llama-index-core                      # 核心：索引 / 查询 / 内存 / 工具等
!pip install -U llama-index-readers-file llama-index-readers-web   # 数据连接器：读本地文件 / 网页

# —— 模型：阿里云百炼 DashScope（千问 LLM + Embedding 向量化）——
!pip install -U llama-index-llms-dashscope llama-index-embeddings-dashscope

# —— 向量库与检索器 ——
!pip install -U llama-index-vector-stores-faiss faiss-cpu   # FAISS 向量库（存向量、做相似度搜索）
!pip install -U llama-index-retrievers-bm25 rank-bm25       # BM25 关键词检索（混合检索用）

# —— 文档解析：本地 PDF / Word / PPT / 网页 ——
!pip install -U pypdf python-pptx trafilatura          # PDF / PPT / 网页正文提取

# —— 云解析服务：复杂 PDF（表格/公式/扫描件）走云端，识别更准 ——
!pip install -U llama-parse

# —— 可选（按需取消注释）——
# !pip install -U sentence-transformers    # 离线/本地 embedding 模型
# !pip install -U paddleocr pillow         # 扫描件 OCR（很重，不常用别装）

ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^
ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^
ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^
ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


LlamaIndex 是一个用于 LLM 应用的数据框架，用于注入、结构化并访问私有或特定领域数据，是目前最适合做 RAG 的框架之一。

核心模块：
- **Data connectors**：从各种数据源（数据库、API、文件系统、网页）提取和加载数据。
- **Data indexes**：对文档分块、向量化、提取元信息，建立索引以支持高效检索。
- **Query Engines**：执行查询处理、语义搜索、RAG 检索增强生成。
- **Data agents**：结合 LLM 与工具链自动执行复杂任务（检索、总结、推理）。
- **Application integrations**：与外部应用（LLM、向量库、BI、SaaS）集成。

> 版本说明：2026 年 LlamaIndex 主版本已演进到 **0.12–0.14.x**，当前推荐核心 API 为 `Settings`、`IngestionPipeline`、`AgentWorkflow`。
>
> 模型说明：本教程使用**阿里云百炼 DashScope 的千问**（在线 API）。如已开通，直接填 Key 即可；若想换回 OpenAI，把 `Settings.llm / Settings.embed_model` 换成对应类即可。

## 1. 配置 API Key 与全局模型（阿里云百炼 DashScope 千问）

推荐用 `getpass` 交互输入 API Key，不要写死在代码里；并用 `Settings` 统一配置 LLM 与 Embedding。

> 开通方式：阿里云百炼控制台（https://bailian.console.aliyun.com/）→ 开通服务 → 右上角「API-KEY」创建。
> 环境变量名用 `DASHSCOPE_API_KEY`。

In [12]:
# ══════════════════════════════════════════════════════════
# ══ 【cell-04】配置 API Key + 千问模型 (Settings) ══
# ══════════════════════════════════════════════════════════
# ============ 1. 配置 API Key 与全局模型（阿里云百炼 DashScope 千问）============

import os
import getpass

# —— 1) 准备 API Key ——
# 优先从环境变量读（.env / 系统变量里设过就直接用，不弹框）
if not os.getenv('DASHSCOPE_API_KEY'):
    # 环境变量没有 → 交互式弹框让你粘贴（不会显示在屏幕上，也不写进代码）
    os.environ['DASHSCOPE_API_KEY'] = getpass.getpass('请输入你的阿里云百炼 DashScope API Key: ')
api_key = os.environ['DASHSCOPE_API_KEY']
# 统一取出，后面传给模型用

# —— 2) 导入 LlamaIndex 全局配置 + DashScope 模型 ——
from llama_index.core import Settings
# Settings：LlamaIndex 的「全局配置中心」。
# 在这里设的 llm / embed_model，后续所有索引、检索、问答都会自动沿用，不用每个地方重复传。

from llama_index.llms.dashscope import DashScope
# 千问生成模型（LLM）：负责回答用户问题
from llama_index.embeddings.dashscope import DashScopeEmbedding
# 千问向量模型（Embedding）：把文本变成数字向量，用于相似度检索

# —— 3) 配置生成模型（LLM）——
Settings.llm = DashScope(
    api_key=api_key,              # 必须显式传，DashScope 不读 DASHSCOPE_API_KEY
    model_name='qwen-plus',       # 模型名：qwen-plus(均衡) / qwen-max(最强) / qwen-turbo(最快省)
    temperature=0.2,              # 随机性：越小回答越稳定严谨，0.2 适合事实问答
    is_function_calling_model=False,  # ★ 关键：关掉「函数调用」模式
)
# 为什么关 is_function_calling_model：
#   - 默认 True 时，AgentWorkflow 会自动选 FunctionAgent，依赖千问返回「规范 JSON 的 tool-call 参数」；
#   - 但千问返回的 arguments 经常不干净（夹带文字/换行），json.loads 解析失败 → JSONDecodeError；
#   - 设 False 后，AgentWorkflow 自动改用 ReActAgent（用文本格式解析工具调用），
#     对千问兼容性远好于 FunctionAgent，也是千问官方推荐做法。

# —— 4) 配置向量模型（Embedding）——
Settings.embed_model = DashScopeEmbedding(
    api_key=api_key,              # 同样显式传
    model_name='text-embedding-v3',  # 中文向量模型
    embed_batch_size=10,          # 每次最多发 10 条，DashScope 接口单次上限就是 10
)

print('✅ 已配置 DashScope 千问：', Settings.llm.model_name, '/', Settings.embed_model.model_name)

✅ 已配置 DashScope 千问： qwen-plus / text-embedding-v3


## 2. 最小 RAG 示例（官方最简写法）


In [13]:
# ══════════════════════════════════════════════════════════
# ══ 【cell-06】最小 RAG 示例 ══
# ══════════════════════════════════════════════════════════
# ============ 2. 最小 RAG 示例（官方最简写法）============
# RAG 全流程就 4 步：读文档 → 建索引 → 建问答引擎 → 提问

from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
# SimpleDirectoryReader：读一个目录下所有支持的文件（PDF/Word/PPT/等）
# VectorStoreIndex：把文档切成小块、向量化、建成可检索的索引

# —— 1) 读取文档 ——
documents = SimpleDirectoryReader('data').load_data()
# 遍历 ./data 目录所有文件，解析成 Document 列表
print(f'共加载 {len(documents)} 个文档')

# —— 2) 构建向量索引 ——
index = VectorStoreIndex.from_documents(documents)
# 内部自动完成：分块(chunk) → 用 Settings.embed_model 向量化 → 建索引

# —— 3) 构建查询引擎 ——
query_engine = index.as_query_engine(similarity_top_k=5)
# 返回前 5 个最相关的块作为上下文，交给 LLM 回答

# —— 4) 提问 ——
response = query_engine.query('收盘价多少')
# 流程：问题向量化 → 在索引里找最相似的块 → 把块 + 问题一起给千问 → 生成回答
print(response)

共加载 73 个文档
36.37元


## 3. Data connectors（数据连接器）

`SimpleDirectoryReader` 对表格、图片、复杂版式的解析效果有限，推荐按需选择更专业的读取器：
- 本地文件 → `llama-index-readers-file`（含 `PyMuPDFReader` 等）
- 网页 → `llama-index-readers-web`（`BeautifulSoupWebReader` 等）
- 复杂 PDF → **LlamaParse** 云解析服务（表格 / 公式 / 图片识别更准）

更多读取器见 [llamahub.ai](https://llamahub.ai/?tab=readers)。


In [14]:
# ══════════════════════════════════════════════════════════
# ══ 【cell-08】Data connectors: 本地文件+网页加载 ══
# ══════════════════════════════════════════════════════════
# ============ 3. Data connectors：本地文件 + 网页加载 ============

from llama_index.core import SimpleDirectoryReader
# 通用目录读取器：按文件扩展名选择解析器
from llama_index.readers.file import PyMuPDFReader
# PDF 专用解析器（基于 PyMuPDF/fitz），带文本坐标信息
from llama_index.readers.web import BeautifulSoupWebReader
# 网页解析器：抓取 URL 正文（基于 BeautifulSoup）

# —— 1) 本地文件加载器 ——
local_loader = SimpleDirectoryReader(
    input_dir='./data',                          # 读取目录
    required_exts=['.pdf', '.docx', '.pptx', '.epub', '.md'],  # 只读这些类型的文件
    file_extractor={'.pdf': PyMuPDFReader()},    # 指定 .pdf 用 PyMuPDFReader 解析（带坐标）
)

# —— 2) 网页加载器 ——
web_loader = BeautifulSoupWebReader()

# —— 3) 真正执行加载 ——
local_docs = local_loader.load_data()
# 解析 ./data 下所有指定类型的文件 → 得到本地文档列表
web_docs = web_loader.load_data(urls=[
    'https://cloud.tencent.com/developer/article/2499999'   # 抓取这个网页的正文
])
# 抓取并解析网页 → 得到网页文档列表

# —— 4) 合并两类文档 ——
documents = local_docs + web_docs
# 合并成一个列表，后续分块/索引统一处理
print(f'本地 {len(local_docs)} 篇 + 网页 {len(web_docs)} 篇')

本地 73 篇 + 网页 1 篇


### 3.1 复杂 PDF：LlamaParse（云解析服务）

> 需先在 [https://cloud.llamaindex.ai](https://cloud.llamaindex.ai) 注册并获取 API Key。新版本推荐直接 `from llama_parse import LlamaParse`（旧写法 `from llama_cloud_services import LlamaParse` 也兼容，但 `llama-parse` 包是官方当前入口）。


In [15]:
# ══════════════════════════════════════════════════════════
# ══ 【cell-10】3.1 LlamaParse 云解析 ══
# ══════════════════════════════════════════════════════════
# ============ 3.1 复杂 PDF：LlamaParse 云解析 ============

import nest_asyncio
nest_asyncio.apply()
# 上面两行：LlamaParse 内部用 asyncio 发网络请求，
# 但 Jupyter 已经跑在一个事件循环里，再开异步会报 "loop already running"。
# nest_asyncio.apply() 允许事件循环嵌套，让 async 代码在 notebook 里能跑。
# （后面 cell-14 / cell-25 用 await 也要靠它）

import os
import getpass

# —— 准备 LlamaParse 的 API Key（和 cell-04 同样的兜底逻辑）——
if not os.getenv('LLAMA_CLOUD_API_KEY'):
    os.environ['LLAMA_CLOUD_API_KEY'] = getpass.getpass('请输入你的 LlamaParse API Key (llx-...): ')
llama_key = os.environ['LLAMA_CLOUD_API_KEY']
# 优先读环境变量；没有就弹框输入，不写死明文、不会 KeyError

from llama_parse import LlamaParse
# 导入 LlamaIndex 官方的云文档解析服务。
# 作用：把复杂 PDF（表格 / 公式 / 多栏 / 扫描件）解析成结构化 Markdown，
#       比本地 PyMuPDFReader 识别更准，但要联网 + 消耗 LlamaCloud 额度。

parser = LlamaParse(
    api_key=llama_key,            # 从变量读 Key（上面已就绪）
    result_type='markdown',       # 解析结果输出 Markdown，保留标题 / 表格 / 层级
    num_workers=3,                # 并行解析 3 个文档，加速
    verbose=True,                 # 打印解析进度日志，方便排错
    # language='ch_sim',          # 可选：中文文档显式指定语言
)

file_extractor = {'.pdf': parser}
# 文件扩展名 -> 解析器 的映射。
# 意思是：目录里 .pdf 文件都用 LlamaParse 解析（覆盖默认 PDF 解析器）

documents_cloud = SimpleDirectoryReader(
    './data', file_extractor=file_extractor   # 读 ./data 目录，.pdf 走 LlamaParse
).load_data()
# 遍历 ./data 下所有文件，按 file_extractor 规则解析，返回 Document 列表。
# 结果存到 documents_cloud，可和本地解析的 documents 合并后一起喂给索引。

C:\Users\Administrator\AppData\Local\Temp\ipykernel_35164\1574023559.py:22: DeprecationWarning: The 'llama-parse' package is deprecated and will no longer receive updates. Please migrate to the new unified SDK. See https://developers.llamaindex.ai/python/cloud/llamaparse/getting_started/ and https://github.com/run-llama/llama-cloud-py/blob/main/README.md for migration instructions.
  from llama_parse import LlamaParse


Error while parsing the file '<bytes/buffer>': Illegal header value b'Bearer '
Error while parsing the file '<bytes/buffer>': Illegal header value b'Bearer '
Error while parsing the file '<bytes/buffer>': Illegal header value b'Bearer '


## 4. Data indexes（数据索引）

### 4.1 Chunking（分块）

> 注意：LlamaIndex 0.10.21+ 起参数名从 `overlap` 改为 `chunk_overlap`。


In [16]:
# ══════════════════════════════════════════════════════════
# ══ 【cell-12】4.1 Chunking 分块 ══
# ══════════════════════════════════════════════════════════
# ============ 4.1 Chunking：文档分块 ============
# 为什么分块：整篇文档直接塞给 LLM 太长/超预算，
# 先切成小段(chunk)，检索时只取最相关的几段给 LLM。

from llama_index.core.node_parser import SentenceSplitter
# SentenceSplitter：按句子/段落边界切分文档的分块器（不会把一句话劈成两半）

splitter = SentenceSplitter(
    chunk_size=1024,             # 每块目标长度约 1024 字符（太大丢精度，太小丢上下文）
    chunk_overlap=100,           # 相邻两块重叠 100 字符，防止切断关键信息
    paragraph_separator='\n\n',  # 段落分隔符，让表格/段落尽量整块保留
    # secondary_chunking_regex='<table>(.+?)</table>',  # 可选：HTML 里让表格整体保留
)

# —— 把文档列表切成节点(node)列表 ——
nodes = splitter.get_nodes_from_documents(documents)
# nodes：切好的小块，每个带文本 + 元信息，是后续向量化和检索的基本单位

avg = sum(len(n.text) for n in nodes) // max(len(nodes), 1)
# 算平均块长度，方便了解切分结果
print(f'切分成 {len(nodes)} 个 chunk，平均长度约 {avg} 字')

切分成 104 个 chunk，平均长度约 692 字


### 4.2 现代做法：IngestionPipeline（分块 + 向量化 一条流水线）

需要清洗、去重、抽取元数据、统一向量化的场景，推荐用 `IngestionPipeline`，可自由组合多个环节。


In [17]:
# ══════════════════════════════════════════════════════════
# ══ 【cell-14】4.2 IngestionPipeline 流水线 ══
# ══════════════════════════════════════════════════════════
# ============ 4.2 IngestionPipeline：分块 + 向量化 一条流水线 ============
# 现代做法：把「清洗 → 分块 → 向量化 → 存库」串成一条流水线，
# 便于统一管理、复用和加环节（去重、抽元数据等）。

from llama_index.core.ingestion import IngestionPipeline
# 数据摄入流水线：按顺序对文档执行一系列变换
from llama_index.core.node_parser import SentenceSplitter
# 分块器（见 4.1）

# —— 定义流水线：文档 → 分块 → 向量化 ——
pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=1024, chunk_overlap=100),  # 第1步：切块
        Settings.embed_model,                                   # 第2步：向量化（用全局配置的千问向量模型）
    ],
)
# 想加环节就继续往列表里追加，比如：
#   TitleExtractor()        # 抽标题
#   MetadataExtractor()     # 抽元信息
#   IngestionCache          # 缓存避免重复计算

# —— 异步执行流水线 ——
nodes_pipe = await pipeline.arun(documents=documents)
# arun = async run，异步跑完每个文档的分块+向量化。
# 需要前面 cell-10 的 nest_asyncio 支持。
print(f'IngestionPipeline 产出 {len(nodes_pipe)} 个节点（已含向量）')

IngestionPipeline 产出 104 个节点（已含向量）


### 4.3 Indexing：FAISS 向量库（比内置内存索引更快、更可扩展）

> 向量维度必须与 embedding 模型一致：`text-embedding-v3` → **1024**（换回 OpenAI `text-embedding-3-small` → 1536，`-3-large` → 3072，需同步改 `d`）。

In [18]:
# ══════════════════════════════════════════════════════════
# ══ 【cell-16】4.3 Indexing: FAISS 建索引 ══
# ══════════════════════════════════════════════════════════
# ============ 4.3 Indexing：FAISS 向量库建索引 ============
# 内置内存索引数据量大时检索慢，FAISS 是专业向量库，更快、可扩展。

import faiss
# FAISS：Facebook 开源的高性能向量相似度搜索库
from llama_index.core import StorageContext, VectorStoreIndex
# StorageContext：管理「向量存哪 + 节点信息存哪」的上下文
# VectorStoreIndex：文档索引的载体
from llama_index.core.retrievers import VectorIndexRetriever
# 向量检索器：给定查询，在索引里找最相似的块
from llama_index.core.query_engine import RetrieverQueryEngine
# 查询引擎：把「检索器」包装成可问答的引擎
from llama_index.vector_stores.faiss import FaissVectorStore
# LlamaIndex 对 FAISS 的适配：让 FAISS 当 LlamaIndex 的向量存储后端

# —— 1) 创建 FAISS 索引 ——
d = 1024  # 向量维度 = embedding 维度（text-embedding-v3 → 1024）
faiss_index = faiss.IndexFlatL2(d)
# IndexFlatL2：最简单的暴力检索索引，用 L2 距离衡量相似度
vector_store = FaissVectorStore(faiss_index=faiss_index)
# 包成 LlamaIndex 能用的向量存储

# —— 2) 存储上下文 ——
storage_context = StorageContext.from_defaults(vector_store=vector_store)
# 告诉 LlamaIndex：向量存到上面的 FAISS，节点信息走默认路径

# —— 3) 用 FAISS 构建向量索引 ——
vector_index = VectorStoreIndex(nodes, storage_context=storage_context)
# 把前面切好的 nodes 灌进 FAISS，建立可检索的索引

# —— 4) 创建向量检索器 ——
vector_retriever = VectorIndexRetriever(
    index=vector_index,
    similarity_top_k=5,   # 每次召回最相似的 5 块
)

# —— 5) 构建查询引擎 ——
query_engine = RetrieverQueryEngine.from_args(retriever=vector_retriever)
# 把检索器包装成查询引擎：检索 + 拼上下文 + 交给 LLM 回答

# —— 6) 执行查询 ——
response = query_engine.query('同仁堂安宫牛黄丸的市场价格，中文回答')
print(response)

2026-08-02 08:27:58,056 - INFO - Loading faiss with AVX2 support.
2026-08-02 08:27:58,077 - INFO - Successfully loaded faiss with AVX2 support.


截至2024年6月，同仁堂双天然安宫牛黄丸（3g/丸）在内地的零售价格为860元/丸；在香港市场的最新零售价为1280元/丸。该产品自2012年以来在内地已提价3次，而在香港自2010年以来提价超8次，2024年6月的提价幅度达20%。


### 4.4 检索增强：混合检索（向量 + BM25）+ Rerank 重排

只靠向量检索容易漏掉精确关键词匹配；加上 BM25 做混合召回、再用 Rerank 精排，是当前最常用的 RAG 进阶配置。


In [19]:
# ══════════════════════════════════════════════════════════
# ══ 【cell-18】4.4 混合检索+BM25+Rerank ══
# ══════════════════════════════════════════════════════════
# ============ 4.4 混合检索（向量 + BM25）+ Rerank 重排 ============
# 问题：纯向量检索容易漏掉「关键词精确匹配」（如"安宫牛黄丸"）。
# 方案：向量检索 + 关键词检索 各召回一批，融合后交给 Rerank 精排，取最相关的。

from llama_index.retrievers.bm25 import BM25Retriever
# BM25：经典关键词检索算法（像搜索引擎那样按词频/重要性打分）
from llama_index.core.retrievers import QueryFusionRetriever
# 混合检索器：把多个检索器的结果融合
from llama_index.core.postprocessor import LLMRerank
# LLM 重排器：让 LLM 对召回结果重新打分，留最相关的
from llama_index.core.query_engine import RetrieverQueryEngine

# —— 1) 关键词检索器（BM25）——
bm25_retriever = BM25Retriever.from_defaults(
    nodes=nodes,            # 在之前切好的块上建立关键词索引
    similarity_top_k=5,     # 召回最相关的 5 块
)

# —— 2) 混合检索：向量 + 关键词 融合 ——
hybrid_retriever = QueryFusionRetriever(
    [vector_retriever, bm25_retriever],  # 向量检索器 + BM25 检索器
    similarity_top_k=5,      # 融合后取 top5
    num_queries=1,           # 不对查询做多角度改写，就设 1（想增强可改 >1）
)

# —— 3) Rerank：LLM 精排 ——
reranker = LLMRerank(top_n=3, llm=Settings.llm)
# 让千问在融合结果里再挑一遍，只留最相关的 3 块
# （中文场景可换开源 CrossEncoderRerank + bge-reranker-base，更快更省）

# —— 4) 组装查询引擎 ——
query_engine = RetrieverQueryEngine.from_args(
    retriever=hybrid_retriever,      # 用混合检索器
    node_postprocessors=[reranker],  # 检索后再做 LLM 重排
)

# —— 5) 查询 ——
response = query_engine.query('世运电路2023年同比增长率是多少？中文回答')
print(response)

2026-08-02 08:28:10,971 - DEBUG - Building index from IDs objects


世运电路2023年营业收入同比增长率为10.1%，归母净利润同比增长率为15.1%。


### 4.5 调试：查看检索结果

检索效果不好时，先看召回结果，再决定是调 chunk 大小还是换检索 / Rerank 策略。


In [20]:
# ══════════════════════════════════════════════════════════
# ══ 【cell-20】4.5 调试: 查看检索结果 ══
# ══════════════════════════════════════════════════════════
# ============ 4.5 调试：查看检索结果 ============
# 目的：检索效果不好时，先肉眼看召回的是什么，再决定调 chunk / 检索策略 / Rerank。

# —— 直接用检索器召回，不经过 LLM ——
retrieved_nodes = vector_retriever.retrieve('同仁堂安宫牛黄丸的市场价格，中文回答')
# retrieve()：只做检索（向量找相似），返回排序后的节点列表，带相似度得分

print(f'共检索到 {len(retrieved_nodes)} 个节点')
for i, node in enumerate(retrieved_nodes):
    print(f'\n节点 {i+1}:')
    print(f'相似度得分: {node.score}')              # 和问题的相似度（越小越近，L2 距离）
    print(f'内容: {node.node.text[:300]}')           # 只打印前 300 字，方便快速浏览
    print('-' * 50)

共检索到 5 个节点

节点 1:
相似度得分: 0.3838094174861908
内容: 2025-03-06 同仁堂 
 
 
敬请参阅最后一页免责声明 
-20- 
证券研究报告 
 
图22： 名贵中药材天然麝香实物图 
图23： 名贵中药材天然牛黄/胆黄实物图 
 
 
资料来源：康美中药网，诚通证券研究所 
资料来源：百度百科，诚通证券研究所 
 
3.4、 销售以零售渠道为主，拥有自主定价能力 
公司产品约90%通过院外零售渠道销售，医院渠道销售占比仅10%左右，渠道
模式以院外零售药店销售为主，结合线上销售和少量院内销售。 
近年来公司主力产品安宫牛黄丸在内地和香港市场持续提价，拥有较强自主定
价能力。从2012 年以来，内地安宫牛黄丸（双天然，3g/丸）提价3 次，
--------------------------------------------------

节点 2:
相似度得分: 0.39625751972198486
内容: 2025-03-06 同仁堂 
 
 
敬请参阅最后一页免责声明 
-14- 
证券研究报告 
 
图12： 2019-2023 年安宫牛黄丸零售市场规模及增速 
图13： 2023 年安宫牛黄丸三大终端市场格局 
资料来源：中康CHIS 数据，观研天下，诚通证券研究所 
资料来源：米内网，诚通证券研究所 
安宫牛黄丸国内市场呈现“一超多强”的竞争格局。国内拥有安宫牛黄丸生产
批文的企业超过120 家，目前我国安宫牛黄丸主要品牌有北京同仁堂、南京同仁堂、
广誉远、宏济堂、马百良（香港）、中一药业（白云山）、九芝堂、达仁堂、胡庆
余堂等。根据安宫牛黄丸主要中药材原料牛黄和麝香的来源，安宫牛黄丸可
--------------------------------------------------

节点 3:
相似度得分: 0.4441258907318115
内容: 6
9.7
10.4
12.8
33.7%
45.9%
7.3%
22.2%
0.0%
5.0%
10.0%
15.0%
20.0%
25.0%
30.0%
35.0%
40.0%
45.0%
50.0%
0.0
10.0
20.0
30.0
40.0
50.0
60.0
2019年
2020年
2021年
2022年
2023年
2019-2

## 5. Query Engines（带记忆的聊天）

> 修复：旧示例在聊天里引用了未定义变量 `index`，这里统一改用上面的 `vector_index`。


In [21]:
# ══════════════════════════════════════════════════════════
# ══ 【cell-22】5. Query Engines 带记忆聊天 ══
# ══════════════════════════════════════════════════════════
# ============ 5. Query Engines：带记忆的聊天 ============
# 用途：让 RAG 支持多轮对话，能记住之前问过什么（基于上面 FAISS 建好的 vector_index）。

from llama_index.core.memory import ChatMemoryBuffer
# 聊天记忆缓冲：自动保存/裁剪对话历史，超限自动丢弃最老的

memory = ChatMemoryBuffer.from_defaults(token_limit=5000)
# 记忆上限 5000 token，超过就把最老的对话丢掉

# —— 构建带记忆的聊天引擎 ——
chat_engine = vector_index.as_chat_engine(
    chat_mode='context',        # 模式：每轮先检索相关文档，再结合对话历史回答
    memory=memory,              # 注入记忆
    system_prompt=(             # 系统提示词：约束 AI 的角色和行为
        '你是一个聊天机器人。'
        '要优先基于检索回来的信息回答问题；'
        '如果没有检索到相关信息，根据你自身的能力回答，不要不回答问题。'
    ),
)

# —— 多轮对话 ——
r1 = chat_engine.chat('詹姆斯是谁')
r2 = chat_engine.chat('世运电路2023年上半年实现营业收入多少')
print(r1)
print('-' * 130)
print(r2)
r3 = chat_engine.chat('你可以告诉我更多吗')   # 第三轮会带上前面对话的记忆
print(r3)

根据当前提供的上下文信息，没有关于“詹姆斯”的任何相关内容。因此无法确定您所指的“詹姆斯”是谁（例如是否是某位公众人物、作者、报告中提及的人物，或与文档内容相关的角色）。

如果您能提供更多背景信息（如全名、相关领域、上下文场景等），我可以更好地帮您解答！
----------------------------------------------------------------------------------------------------------------------------------
根据提供的研报内容，世运电路2023年上半年实现营业收入**21.51亿元**。
抱歉，当前提供的上下文材料（即《2024年大模型产业发展报告.pdf》）中**并未包含关于“世运电路”的任何信息**，包括其营业收入、业务范围、财务数据或相关分析。

您提到的“世运电路2023年上半年实现营业收入21.51亿元”这一数据**不在本知识库的检索范围内**，因此我无法确认其来源是否准确，也无法提供进一步的细节（如同比变化、毛利率、分产品/地区收入、净利润等）。

如果您有该公司的财报原文、公告链接，或希望我基于公开渠道（如巨潮资讯网、公司年报）协助查找和解读，请告知，我可以指导您如何获取或帮您分析——但需注意：我当前无法实时联网，所有回答必须基于您提供的资料或已加载的上下文。

是否需要我帮您梳理如何查找上市公司财报，或解析财报关键指标？


## 6. Data agents（AgentWorkflow，新版）

> 旧版 `llm.predict_and_call` 已废弃，2026 年推荐用 **`AgentWorkflow`** 编排工具调用：
> 一个 Agent = LLM + 一组工具 + 系统提示词，由模型自动决定何时调用哪个工具，支持多工具链式执行。


In [22]:
# ══════════════════════════════════════════════════════════
# ══ 【cell-24】6. 定义 Agent 工具 ══
# ══════════════════════════════════════════════════════════
# ============ 6. Data agents：定义 Agent 的工具 ============
# Agent = LLM + 一组工具 + 系统提示词，模型自动决定调用哪个工具。
# 这里给 Agent 准备两个工具：向量检索(具体问题) + 全文总结(整体概述)。

import nest_asyncio
nest_asyncio.apply()   # 允许 asyncio 嵌套，AgentWorkflow 内部异步要用的

from typing import List
from llama_index.core.agent.workflow import AgentWorkflow
# AgentWorkflow：新版 Agent 编排框架（替代已弃用的 llm.predict_and_call）
from llama_index.core.tools import FunctionTool
# FunctionTool：把一个 Python 函数包装成「Agent 可调用的工具」
from llama_index.core.vector_stores import MetadataFilters, FilterCondition
# 元数据过滤：按文档属性(如页码)筛检索范围

# —— 工具 1：向量检索 ——
def vector_query(query: str, page_numbers: List[str]) -> str:
    """在文档向量索引上执行检索。

    参数:
        query: 要检索的查询字符串。
        page_numbers: 按页码过滤；传空列表表示检索全部页。
    """
    if page_numbers:
        # 有指定页码 → 构造元数据过滤条件
        metadata_dicts = [{'key': 'page_label', 'value': p} for p in page_numbers]
        filters = MetadataFilters.from_dicts(
            metadata_dicts, condition=FilterCondition.OR   # OR：任一页码匹配即可
        )
    else:
        filters = None   # 没指定 → 不过滤，检索全部页

    qe = vector_index.as_query_engine(similarity_top_k=5, filters=filters)
    return str(qe.query(query))   # 执行检索+回答，返回文本


vector_query_tool = FunctionTool.from_defaults(
    name='vector_tool',                                   # 工具名（Agent 会按名调用）
    description='在文档向量索引上做精确检索，回答具体数据问题时使用',  # 告诉 Agent 何时用它
    fn=vector_query,                                      # 绑定的函数
)

# —— 工具 2：全文总结 ——
from llama_index.core import SummaryIndex
# SummaryIndex：把节点按顺序排列的索引，适合做全文总结
from llama_index.core.tools import QueryEngineTool
# QueryEngineTool：把一个查询引擎包装成工具

summary_index = SummaryIndex(nodes)
# 在切好的 nodes 上建总结索引
summary_query_engine = summary_index.as_query_engine(response_mode='tree_summarize')
# tree_summarize：把多块内容分层汇总，适合长文档总结
summary_tool = QueryEngineTool.from_defaults(
    name='summary_tool',
    query_engine=summary_query_engine,
    description='需要对整篇文章做全局总结时使用',   # 告诉 Agent 何时用它
)

In [23]:
# ══════════════════════════════════════════════════════════
# ══ 【cell-25】6(续) 运行 Agent ══
# ══════════════════════════════════════════════════════════
# ============ 6(续) Data agents：运行 Agent ============

# —— 用两个工具 + 千问 构建单个 Agent ——
agent = AgentWorkflow.from_tools_or_functions(   # 0.14 版 API 名（from_tools 已改名）
    tools_or_functions=[vector_query_tool, summary_tool],  # 给 Agent 配的工具列表
    llm=Settings.llm,                          # 用全局配置的千问
    system_prompt=(                            # 告诉 Agent 自己的职责和分工
        '你是 RAG 助手。'
        '需要具体数据时调用 vector_tool，需要全文总结时调用 summary_tool。'
    ),
    verbose=True,                              # 打印 Agent 每一步的思考/调用过程，方便观察
)

# —— 异步运行 ——
resp = await agent.run('总结一下兴证电子')
# agent.run：Agent 自动判断该调用哪个工具（这里是 summary_tool 做总结）
print(resp)

resp2 = await agent.run('世运电路2023年同比增长情况')
# 这个是具体数据问题 → 会调用 vector_tool 检索文档
print(resp2)

2026-08-02 08:29:41,107 - INFO - [tick] add: AgentWorkflowStartEvent(user_msg='总结一下兴证电子', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
2026-08-02 08:29:41,108 - INFO - [init_run:0] started from AgentWorkflowStartEvent
2026-08-02 08:29:42,204 - INFO - [init_run:0] complete with AgentInput
2026-08-02 08:29:42,205 - INFO - [tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='总结一下兴证电子')])], current_agent_name='Agent')
2026-08-02 08:29:42,206 - INFO - [setup_agent:0] started from AgentInput
2026-08-02 08:29:42,208 - INFO - [setup_agent:0] complete with AgentSetup
2026-08-02 08:29:42,209 - INFO - [tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.SYSTEM: 'system'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='你是 RAG 助手。需要具体数据时调用 vector_tool，需要全文总结时调用 summary_tool。')]), Chat...
2026-08-02 08:29:42,209 - INFO - [run_agent_step:0] started from A

兴证电子是兴业证券股份有限公司研究所下属的电子行业研究团队，专注于电子领域上市公司的深度研究与投资价值评估。该团队定期发布行业及公司点评报告，例如对世运电路（603920）2023年中报的分析，内容涵盖财务表现、业务进展、盈利预测、投资建议及风险提示，并维持“增持”评级。其研究报告基于公开信息与专业研判，服务于机构及专业投资者，强调独立性、客观性和合规性；同时明确提示报告观点不构成对所有投资者的统一投资建议，且兴业证券可能存在与报告观点不一致的自营或资产管理操作。


2026-08-02 08:29:53,787 - INFO - [init_run:0] complete with AgentInput
2026-08-02 08:29:53,788 - INFO - [tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='世运电路2023年同比增长情况')])], current_agent_name='Agent')
2026-08-02 08:29:53,788 - INFO - [setup_agent:0] started from AgentInput
2026-08-02 08:29:53,791 - INFO - [setup_agent:0] complete with AgentSetup
2026-08-02 08:29:53,792 - INFO - [tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.SYSTEM: 'system'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='你是 RAG 助手。需要具体数据时调用 vector_tool，需要全文总结时调用 summary_tool。')]), Chat...
2026-08-02 08:29:53,792 - INFO - [run_agent_step:0] started from AgentSetup
2026-08-02 08:29:55,923 - INFO - [run_agent_step:0] complete with AgentOutput
2026-08-02 08:29:55,924 - INFO - [tick] add: AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={'tool_calls': []}, 

世运电路2023年（预测值）同比增长情况如下：

- 营业收入同比增长10.1%；  
- 归母净利润同比增长15.1%；  
- 营业利润同比增长27.7%；  
- 毛利率同比提升0.8个百分点（由2022年的18.8%升至19.6%）；  
- 净资产收益率（ROE）同比提升1.5个百分点（由2022年的14.5%升至16.0%）。  

注：以上数据为基于2022年实际值的2023年预测（2023E）同比增长率。


## 7. 工程实践：索引持久化与复用

把索引保存到磁盘，避免每次重启都重新解析 + 向量化（对大数据集尤其重要）。


In [25]:
# ══════════════════════════════════════════════════════════
# ══ 【cell-27】7. 索引持久化与复用 ══
# ══════════════════════════════════════════════════════════
# ============ 7. 工程实践：索引持久化与复用 ============
# 目的：把建好的索引存到磁盘，下次不用重新解析+向量化（大数据集省很多时间/费用）。
#
# ⚠️ 注意：这里的「向量存储」必须是 JSON 可加载的类型（默认 SimpleVectorStore）。
#     FAISS 后端持久化是二进制，不能直接 load_index_from_storage（会 UnicodeDecodeError）。
#     所以下面重新用默认向量存储建一个持久化索引来演示。

# —— 1) 用默认向量存储（SimpleVectorStore）新建索引 ——
from llama_index.core import VectorStoreIndex
# VectorStoreIndex 默认后端 = SimpleVectorStore（纯 JSON 持久化）
index_persist = VectorStoreIndex(nodes)
# 注意：nodes 要在内存里。若重启了内核，需先重跑 cell-12（分块）拿到 nodes

# —— 2) 保存索引 ——
index_persist.storage_context.persist(persist_dir='./storage')
# 保存到 ./storage，含向量 + 节点信息，都是 JSON 格式，可被 load 正确解析

# —— 3) 从磁盘加载 ——
from llama_index.core import StorageContext, load_index_from_storage

storage_context = StorageContext.from_defaults(persist_dir='./storage')
# 指定从 ./storage 恢复
loaded_index = load_index_from_storage(storage_context)
# 加载出索引对象，可直接问答

# —— 4) 用加载的索引问答 ——
query_engine = loaded_index.as_query_engine()
print(query_engine.query('同仁堂安宫牛黄丸的市场价格，中文回答'))

# ⚠️ 注意：加载时 Settings.embed_model 必须和保存时一致，
#    否则向量维度不匹配会报错。

# —— 附：FAISS 索引要持久化怎么存？——
# 因为 FAISS 是二进制，加载要显式用 FaissVectorStore：
#   from llama_index.vector_stores.faiss import FaissVectorStore
#   from llama_index.core import StorageContext, load_index_from_storage
#   import faiss
#   faiss_idx = faiss.read_index('./storage/vector_store.index')  # 需另存 .index 文件
#   vs = FaissVectorStore.from_persist_dir('./storage')           # 或从目录恢复
# 更推荐：直接用默认 SimpleVectorStore 做持久化（本 cell 演示），
#         生产大数据量再上专业向量库（PGVector / Milvus / Qdrant 等）。

2026-08-02 08:31:28,257 - INFO - Loading all indices.


同仁堂安宫牛黄丸（双天然，3g/丸）在内地的最新零售价为860元/丸，该价格系自2012年以来第三次提价；在香港市场，同仁堂国药双天然安宫牛黄丸于2024年6月最新提价至1280元/丸，较此前上涨20%。此外，内地零售价相较香港存在明显差距，且受天然牛黄大幅涨价影响，内地价格仍有进一步上调空间。
